In [19]:
import re
import numpy as np
import faiss
import torch
import pymupdf
import gradio as gr
from hazm import Normalizer
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- مدل‌ها ----------
embed_model = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)

normalizer = Normalizer()
chunks, index = [], None

# ---------- پردازش متن ----------
ZWNJ = "\u200c"
word_fixes = {"خالصه": "خلاصه", "اخالقی": "اخلاقی", "الزم": "لازم", "باالتر": "بالاتر"}

def fix_text(t):
    t = re.sub(ZWNJ + r"،\s*", ZWNJ, t)
    return re.sub(r"[\u0600-\u06FF]+", lambda m: word_fixes.get(m.group(0), m.group(0)), t)

def read_file(path):
    if path.lower().endswith(".pdf"):
        doc = pymupdf.open(path)
        return "\n".join(p.get_text() for p in doc)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def chunk_text(text, chunk_size=200, overlap=40):
    words = text.split()
    step = chunk_size - overlap
    out = []
    for start in range(0, len(words), step):
        c = " ".join(words[start:start + chunk_size])
        if c.strip():
            out.append(c)
        if start + chunk_size >= len(words):
            break
    return out

# ---------- RAG ----------
def build_index(file_path):
    global chunks, index
    if file_path is None:
        return "اول یه فایل آپلود کن."
    t = fix_text(normalizer.normalize(read_file(file_path)))
    chunks = chunk_text(t)
    emb = embed_model.encode(["passage: " + c for c in chunks],
                             normalize_embeddings=True, batch_size=16)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(np.array(emb, dtype="float32"))
    return f"فایل پردازش شد: {len(chunks)} تکه ساخته شد."

def search(question, k=3):
    q = embed_model.encode(["query: " + question], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype="float32"), min(k, len(chunks)))
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0])]

def answer(question, results, max_new_tokens=200):
    context = "\n\n".join(f"[{i+1}] {c}" for i, (c, s) in enumerate(results))
    messages = [
        {"role": "system", "content":
            "تو یک دستیار فارسی‌زبان هستی. فقط بر اساس متن‌های داده‌شده پاسخ بده. "
            "اگر جواب در متن‌ها نبود، بگو «در متن پیدا نکردم»."},
        {"role": "user", "content": f"متن‌ها:\n{context}\n\nسؤال: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def ask(question):
    if index is None:
        return "اول یه فایل آپلود کن.", ""
    if not question.strip():
        return "سؤال خالیه.", ""
    results = search(question)
    sources = "\n\n".join(f"[{i+1}] (شباهت {s:.2f})\n{c[:300]}..." for i, (c, s) in enumerate(results))
    return answer(question, results), sources

# ---------- رابط ----------
with gr.Blocks() as demo:
    gr.Markdown("## چت‌بات RAG روی اسناد فارسی")
    file_in = gr.File(label="فایل PDF یا txt", type="filepath")
    status = gr.Textbox(label="وضعیت", interactive=False)
    file_in.upload(build_index, file_in, status)
    q = gr.Textbox(label="سؤال", rtl=True)
    btn = gr.Button("بپرس")
    a = gr.Textbox(label="جواب", rtl=True, interactive=False)
    src = gr.Textbox(label="منبع‌ها", rtl=True, interactive=False, lines=6)
    btn.click(ask, q, [a, src])
    q.submit(ask, q, [a, src])

demo.launch()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6fea3f6568b9ec08fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
